# Chapter 04: Reading PR Data (Reference)

## Learning Objectives

- Fetch a single PR's raw object and normalize it into `PRMetadata`
- List several PRs with `gh pr list` semantics
- Split a result set into page-sized chunks, mirroring real pagination
- State the two concrete ceilings: the 100-item page cap and the 300-file cap

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path`. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. Fetch and Normalize One PR

The next cell fetches PR #101's raw object and normalizes it into `PRMetadata`. You should see the raw additions/deletions/changed_files line, followed by the normalized dataclass and its `lines_changed` property.

In [ ]:
from labs.lab_04_pr_data import fetch_single_pr, normalize_pr

raw = fetch_single_pr("example/example", 101)
metadata = normalize_pr(raw)
print(f"raw: {raw['additions']}/{raw['deletions']}/{raw['changed_files']}")
print(f"normalized: {metadata}")
print(f"lines_changed: {metadata.lines_changed}")


## 2. List and Paginate

The next cell lists several PRs, then splits them into page-sized chunks with `paginate_in_chunks` -- a miniature of what `gh api --paginate` does automatically by following `Link: rel="next"`. You should see the PRs split into pages of 5.

In [ ]:
from labs.lab_04_pr_data import list_open_prs, paginate_in_chunks, MAX_PAGE_SIZE, MAX_FILES_LISTED

prs = list_open_prs("example/example", limit=100)
print(f"fetched {len(prs)} PRs (page size cap is {MAX_PAGE_SIZE})")

pages = paginate_in_chunks(prs, page_size=5)
for i, page in enumerate(pages, start=1):
    numbers = ", ".join(f"#{p['number']}" for p in page)
    print(f"  page {i}: {numbers}")

print(f"\nfiles-list cap (separate from the page cap): {MAX_FILES_LISTED}")


## Takeaways & Next Steps

This notebook's takeaways are the numbers above: the normalized `PRMetadata`, the page split, and the two distinct ceilings (100-item pages vs the 300-file hard cap).

In [ ]:
print("Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.")


---

📖 **Reading companion:** [Chapter 04: Reading PR Data](../learning_modules/chapter_04_reading_pr_data.md)
